# Diabetes Disease Progression Prediction

**Internship:** Data Science Internship — Aenexz Tech Portals  
**Domain:** Healthcare Analytics & Supervised Machine Learning  
**Dataset:** `load_diabetes()` from Scikit-Learn  

---

## 1. Project Overview & Objective
Diabetes is a chronic metabolic disease that affects millions of individuals worldwide. Timely tracking and prediction of diabetes progression enables healthcare practitioners to adjust treatment plans and dietary recommendations proactively.

### Objective:
Predict the quantitative measure of **disease progression one year after baseline** using ten baseline physiological and serum measurements.

### Feature Definitions:
- **`age`**: Age in years
- **`sex`**: Biological sex
- **`bmi`**: Body Mass Index
- **`bp`**: Average blood pressure
- **`s1`**: Total serum cholesterol (tc)
- **`s2`**: Low-density lipoproteins (ldl)
- **`s3`**: High-density lipoproteins (hdl)
- **`s4`**: Total cholesterol / HDL (tch)
- **`s5`**: Log of serum triglycerides level (ltg)
- **`s6`**: Blood sugar level (glu)

### Target:
- **`target`**: A quantitative measure of disease progression one year after baseline (ranges from 25 to 346).

## Step 1: Import Libraries and Load Dataset
We begin by importing the required numerical, data manipulation, visualization, and modeling libraries.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load scikit-learn diabetes dataset as a pandas DataFrame
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame.copy()

print("Dataset successfully loaded!")
print(f"Dimensions: {df.shape[0]} rows, {df.shape[1]} columns (10 features, 1 target)")
print(f"Features: {diabetes.feature_names}")
df.head()

Dataset successfully loaded!
Dimensions: 442 rows, 11 columns (10 features, 1 target)
Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


### Dataset Architecture Note
In Scikit-Learn's `load_diabetes()`, the raw clinical values have been mean-centered and scaled by the standard deviation times the square root of $n\text{ (samples)}$ ($n=442$). This means each feature column has a mean of approximately $0$ and the sum of squares equals $1$. Because all features are already standard-scaled, their regression coefficients can be directly compared to evaluate relative feature impact without any additional feature scaling.

## Step 2: Exploratory Data Analysis (EDA)
Let's inspect the data types, missing values, and descriptive statistics across all features and the target.

In [2]:
# Check missing values and schema
print(df.info())
print("\nMissing values count per column:")
print(", ".join([f"{col}: {df[col].isnull().sum()}" for col in df.columns]))
df.describe().round(4)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB

Missing values count per column:
age: 0, sex: 0, bmi: 0, bp: 0, s1: 0, s2: 0, s3: 0, s4: 0, s5: 0, s6: 0, target: 0


### EDA Visualizations: Feature and Target Distributions
We visualize the distributions of each feature and the target variable to understand spread, skewness, and potential anomalies.

In [3]:
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.flatten()

feature_cols = diabetes.feature_names
for i, col in enumerate(feature_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#1f77b4', bins=20)
    axes[i].set_title(f'Distribution: {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')

# Target distribution
sns.histplot(df['target'], kde=True, ax=axes[10], color='#d62728', bins=20)
axes[10].set_title('Target: Disease Progression', fontsize=12, fontweight='bold')
axes[10].set_xlabel('Score after 1 year')

axes[11].axis('off')
plt.suptitle('Exploratory Data Analysis: Feature & Target Distributions', fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

### Key Observations from EDA:
1. **No Missing Data**: All 442 records have complete non-null values across all 10 features and the target variable.
2. **Target Distribution**: The progression score spans from 25 to 346 with a mean of 152.13 and median of 140.50. The distribution exhibits positive (right) skewness.
3. **Binary Feature**: The `sex` feature takes two distinct standardized values (-0.0446 and +0.0507), corresponding to male and female categories.
4. **Continuous Features**: Features such as `bmi`, `bp`, `s1`–`s6` exhibit near-normal or mildly skewed continuous distributions.

## Step 3: Feature Correlation Analysis
Understanding linear correlations helps reveal which physiological indicators are most aligned with disease progression, as well as detecting multicollinearity among predictor variables.

In [4]:
# Correlation matrix
corr_matrix = df.corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap of Diabetes Features and Progression Target', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [5]:
# Ranked feature correlation with target
target_corr = corr_matrix['target'].drop('target').sort_values(ascending=False)
print("Correlation of Features with Disease Progression (Target):")
print(target_corr.round(4))

# Visualizing correlation with target
plt.figure(figsize=(9, 5))
colors = ['#2ca02c' if c > 0 else '#d62728' for c in target_corr.values]
sns.barplot(x=target_corr.values, y=target_corr.index, hue=target_corr.index, palette=colors, legend=False)
plt.axvline(0, color='black', linestyle='--', alpha=0.7)
plt.title('Feature Correlation with Disease Progression Target', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

Correlation of Features with Disease Progression (Target):
bmi    0.5865
s5     0.5659
bp     0.4415
s4     0.4305
s6     0.3825
s1     0.2120
age    0.1879
s2     0.1741
sex    0.0431
s3    -0.3948
Name: target, dtype: float64


### Correlation Insights:
- **Strongest Positive Correlates**:
  - **`bmi` (0.586)**: Body mass index has the strongest positive correlation with disease progression, aligning with medical evidence that excess weight exacerbates insulin resistance.
  - **`s5` (0.566)**: Log of serum triglycerides level is another primary positive driver.
  - **`bp` (0.441)**: Higher arterial blood pressure strongly associates with increased disease severity.
  - **`s4` (0.430)**: Total cholesterol / HDL ratio demonstrates notable positive association.
- **Strongest Negative Correlate**:
  - **`s3` (-0.395)**: High-density lipoprotein (HDL "good cholesterol") displays a negative correlation, confirming its protective cardiovascular and metabolic role.
- **Multicollinearity Flag**:
  - Notice strong inter-correlations among serum lipids: `s1` and `s2` have $r = 0.897$, `s3` and `s4` have $r = -0.738$, and `s4` and `s5` have $r = 0.659$. This multicollinearity will be reflected in the regression coefficient signs.

## Step 4: Train Linear Regression Model
Following strict machine learning best practices, we partition the dataset into training (80%) and testing (20%) sets prior to modeling, using a fixed random seed (`random_state=42`) for reproducibility.

In [6]:
# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and fit Ordinary Least Squares Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

# Generate predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("Data Split Completed:")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print("Model Training: LinearRegression() fitted successfully.")
print(f"Model Intercept: {model.intercept_:.4f}")

Data Split Completed:
Training samples: 353
Testing samples: 89
Model Training: LinearRegression() fitted successfully.
Model Intercept: 151.3456


## Step 5: Model Evaluation (R², MAE, RMSE)
We evaluate the model on both the training and testing sets across three standard regression metrics:
1. **Coefficient of Determination ($R^2$)**: Proportion of variance in disease progression explained by the linear combination of predictors.
2. **Mean Absolute Error (MAE)**: Average magnitude of the absolute prediction errors (in target units).
3. **Root Mean Squared Error (RMSE)**: Square root of average squared errors, penalizing larger deviations more heavily.

In [7]:
# Calculate performance metrics
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

metrics_df = pd.DataFrame({
    'Metric': ['R² Score', 'MAE', 'RMSE'],
    'Training Set': [r2_train, mae_train, rmse_train],
    'Testing Set': [r2_test, mae_test, rmse_test]
})

print("=" * 50)
print("           LINEAR REGRESSION MODEL PERFORMANCE     ")
print("=" * 50)
print(f"{'Metric':<18} {'Training Set':<18} {'Testing Set':<16}")
print("-" * 50)
for _, row in metrics_df.iterrows():
    print(f"{row['Metric']:<18} {row['Training Set']:<18.4f} {row['Testing Set']:<16.4f}")
print("=" * 50)

           LINEAR REGRESSION MODEL PERFORMANCE     
Metric             Training Set       Testing Set     
--------------------------------------------------
R² Score                 0.5279            0.4526
MAE                     43.4835           42.7941
RMSE                    53.5588           53.8534


### Evaluation Interpretation:
- **Fit and Generalization**: The testing $R^2$ of **0.4526** (45.3% variance explained) is very close to the training $R^2$ of **0.5279** (52.8%).
- **Error Consistency**: The testing MAE (**42.79**) and testing RMSE (**53.85**) closely match the training values (**43.48** and **53.56**).
- **Overfitting Diagnostic**: There is virtually no severe overfitting; the model demonstrates stable generalization across unseen data.

## Step 6: Identify Most Influential Features
Since all predictor features in the dataset are already standard-scaled with zero mean and unit sum of squares, the magnitude and signs of the regression coefficients indicate their direct impact on predicted disease progression.

In [8]:
# Extract coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_,
    'Abs_Coefficient': np.abs(model.coef_),
    'Direction': np.where(model.coef_ > 0, 'Positive', 'Negative')
}).sort_values(by='Abs_Coefficient', ascending=False)

print("Feature Importance (Ranked by Absolute Coefficient Magnitude):")
print(coef_df.to_string(index=False))

# Visualizing coefficients
plt.figure(figsize=(10, 6))
coef_sorted = coef_df.sort_values(by='Coefficient', ascending=True)
bar_colors = ['#d62728' if c < 0 else '#1f77b4' for c in coef_sorted['Coefficient']]
plt.barh(coef_sorted['Feature'], coef_sorted['Coefficient'], color=bar_colors)
plt.axvline(0, color='black', linestyle='--', alpha=0.7)
plt.title('Linear Regression Coefficients (Direction & Relative Impact)', fontsize=13, fontweight='bold')
plt.xlabel('Regression Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

Feature Importance (Ranked by Absolute Coefficient Magnitude):
Feature  Coefficient  Abs_Coefficient Direction
     s1  -931.488846       931.488846  Negative
     s5   736.198859       736.198859  Positive
    bmi   542.428759       542.428759  Positive
     s2   518.062277       518.062277  Positive
     bp   347.703844       347.703844  Positive
     s4   275.317902       275.317902  Positive
    sex  -241.964362       241.964362  Negative
     s3   163.419983       163.419983  Positive
     s6    48.670657        48.670657  Positive
    age    37.904021        37.904021  Positive


### Feature Importance Insights:
1. **`s1` and `s5` (Lipids / Triglycerides)**:
   - `s1` (Total Serum Cholesterol) has a coefficient of **-931.49** and `s5` (Log of Triglycerides) has a coefficient of **+736.20**.
   - Due to the high collinearity between serum lipids (`s1`, `s2`, `s5`), the model balances the total effect across these variables, making `s5` a primary positive driver of progression.
2. **`bmi` (Body Mass Index)**:
   - Coefficient: **+542.43**. As expected medically, body mass index is one of the strongest and most reliable positive contributors to diabetes progression.
3. **`s2` (LDL Cholesterol)** and **`bp` (Blood Pressure)**:
   - Coefficients of **+518.06** and **+347.70** show that elevated low-density lipoprotein levels and high blood pressure substantially elevate the one-year disease progression score.
4. **Least Influential Features**:
   - `age` (**+37.90**) and `s6` (**+48.67**) have relatively minor weights when conditioned on the other metabolic and serum markers.

## Step 7: Compare Actual vs Predicted Values
To validate the model's predictive behavior on unseen test data, we contrast the actual patient progression scores against the predicted scores both numerically and visually.

In [9]:
# Tabular Comparison for Test Samples
comparison_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_test_pred,
    'Error (Residual)': y_test.values - y_test_pred,
    'Abs_Error': np.abs(y_test.values - y_test_pred)
})

print("Sample Comparison of Actual vs Predicted Progression Scores (Test Set):")
print(comparison_df.head(10).round(2))

# Summary of Prediction Errors
print("\nResiduals Summary:")
print(comparison_df['Error (Residual)'].describe().round(2))

Sample Comparison of Actual vs Predicted Progression Scores (Test Set):
   Actual  Predicted  Error (Residual)  Abs_Error
0   219.0     139.55             79.45      79.45
1    70.0     179.52           -109.52     109.52
2   202.0     134.04             67.96      67.96
3   230.0     291.42            -61.42      61.42
4   111.0     123.79            -12.79      12.79
5    84.0      92.17             -8.17       8.17
6   242.0     258.23            -16.23      16.23
7   272.0     181.34             90.66      90.66
8    94.0      90.22              3.78       3.78
9    96.0     108.63            -12.63      12.63


In [10]:
# Visualizing Actual vs Predicted & Residuals
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Actual vs Predicted
axes[0].scatter(y_test, y_test_pred, color='#1f77b4', edgecolors='k', alpha=0.75, s=65, label='Test Observations')
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Fit Line (y = x)')

# Fit line for observed predictions
m, b = np.polyfit(y_test, y_test_pred, 1)
axes[0].plot(y_test, m*y_test + b, color='navy', linestyle=':', lw=2, label=f'Trend: y={m:.2f}x+{b:.2f}')
axes[0].set_title('Actual vs. Predicted Disease Progression', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Progression Score', fontsize=11)
axes[0].set_ylabel('Predicted Progression Score', fontsize=11)
axes[0].legend(loc='upper left')

# Plot 2: Residuals vs Predicted
residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuals, color='#2ca02c', edgecolors='k', alpha=0.75, s=65)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_title('Residuals vs. Predicted Values', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Progression Score', fontsize=11)
axes[1].set_ylabel('Residual (Actual - Predicted)', fontsize=11)

plt.tight_layout()
plt.show()

### Diagnostic Discussion:
- **Linear Alignment**: Predictions follow the general $y = x$ trajectory across the middle progression range (100–220).
- **Boundary Compression (Regression to Mean)**: For very low progression scores ($<70$), the linear model tends to overpredict, while for extreme high progression scores ($>250$), it tends to underpredict. This is typical for unregularized OLS regression with noisy physiological targets.
- **Residual Pattern**: Residuals are approximately randomly scattered around zero with no extreme heteroscedasticity, confirming that the linear model assumptions hold reasonably well.

## Summary & Conclusion

1. **Dataset Integrity**: The Scikit-Learn `load_diabetes` dataset contains 442 patient observations with 10 standardized clinical and serum features and no missing values.
2. **Exploratory Insights**: Features like `bmi`, `s5` (triglycerides), and `bp` show moderate-to-strong positive correlations with one-year disease progression, while `s3` (HDL "good" cholesterol) shows an expected protective negative correlation.
3. **Model Performance**: The Linear Regression model achieved:
   - **Testing $R^2$**: `0.4526` (~45.3% variance explained on unseen patients)
   - **Testing MAE**: `42.79`
   - **Testing RMSE**: `53.85`
4. **Clinical Key Drivers**: **BMI**, **serum triglycerides (`s5`)**, **LDL cholesterol (`s2`)**, and **blood pressure (`bp`)** were identified as the primary risk factors accelerating diabetes progression.
5. **Recommended Future Enhancements**:
   - Testing regularized linear models (Ridge, Lasso, ElasticNet) to handle multicollinearity among serum measurements (`s1`–`s6`).
   - Exploring non-linear models (Random Forests, Gradient Boosting) to capture non-linear metabolic interactions.